프로토타입
- Notebook 생성 → VSCode에서 notebooks/prototype.ipynb 파일 생성

환경설정
- 프로젝트 전용 가상환경 확인: /Users/ai/deep-learning/LLM/rag_pdf_qdrant/venv/bin/python

필수 라이브러리 설치 및 호환성 확인
- NumPy 1.26.x → 안정적으로 PyTorch 2.2.x와 호환 확인
- PyTorch 2.2.x → 현재 인텔 맥 환경에서 설치 가능한 최신 버전
- SentenceTransformers 2.x~3.x → PyTorch 2.2.x와 호환되는 안정 버전(pip install "sentence-transformers<3"
)
- transformers==4.40.2 정도가 안정적으로 잘 동작(pip install "transformers<4.44"
)

셀 구조 작성
- Import & Qdrant Client 연결
- PDF 불러오기 → 텍스트 추출
- 벡터화 (SentenceTransformers)
- Qdrant 삽입 (client.upsert)
- 검색 (client.search)

테스트 실행 → 간단한 PDF 문서로 end-to-end 확인

In [1]:
# 프로젝트 전용 가상환경 확인: /Users/ai/deep-learning/LLM/rag_pdf_qdrant/venv/bin/python
import sys
print(sys.executable)

/Users/ai/deep-learning/LLM/rag_pdf_qdrant/venv/bin/python


In [2]:
# NumPy 1.26.x → 안정적으로 PyTorch 2.2.x와 호환 확인
import numpy as np
print(np.__version__)

1.26.4


In [3]:
# PyTorch 2.2.x → 현재 인텔 맥 환경에서 설치 가능한 최신 버전
import torch
print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())

2.2.2
CUDA available: False


In [4]:
# SentenceTransformers 2.x~3.x → PyTorch 2.2.x와 호환되는 안정 버전
# SentenceTransformers 모델 로드: 영어 전용
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded")

Model loaded


In [6]:
# SentenceTransformers 모델 로드: 다국어(한국어 포함)
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

In [7]:
# all-MiniLM-L6-v2 모델은 Hugging Face에서 제공하는 경량 SentenceTransformer
# MiniLM 기반으로 설계되어, 각 입력 문장을 384차원 벡터로 변환
# 이 차원 수는 모델 아키텍처에 따라 정해져 있으며, 사용자가 바꿀 수 있는 값이 아니다
# 샘플 벡터 생성
vector = model.encode("Hello Qdrant!").tolist()
print("Vector length:", len(vector))

Vector length: 384


In [8]:
# Qdrant 연결 테스트: Docker container에서 qdrant 서버 실행 후 클라이언트 연결
from qdrant_client import QdrantClient

client = QdrantClient(host="localhost", port=6333)
print(client.get_collections())

collections=[CollectionDescription(name='documents')]


- SentenceTransformer 모델 로드 -> 문장을 벡터화(384차원)
- Qdrant 컬렉션 -> 벡터 크기와 거리(metric)를 지정(COSINE 추천)
- 문서 업로드 -> 각 문서 텍스트를 벡터와 함께 저장, payload에 원문 텍스트도 같이 넣어두면 검색 결과를 쉽게 확인 가능

In [10]:
# 필요 라이브러리 import
from sentence_transformers import SentenceTransformer # 문장을 벡터로 변환하는 트랜스포머 기반 임베딩 모델
from qdrant_client import QdrantClient # Qdrant 서버와 연결하는 클라이언트
from qdrant_client.http import models # Qdrant에서 벡터와 포인트 구조를 정의할때 사용

# 1. 모델 로드: 다국어 지원 모델
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# 2. Qdrant 서버 연결
client = QdrantClient(host="localhost", port=6333)

# 3. Qdrant Collection 생성(없으면 새로 생성)
collection_name = "documents"
# 컬렉션명, 벡터크기 384차원 지정, 거리 계산 방식 COSINE 유사도 사용
client.recreate_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE),
)

# 4. 문서 텍스트 벡터화
texts = [
    "안녕하세요, 이것은 한국어 문서입니다.",
    "Hello, this is an English document.",
    "Qdrant는 벡터 데이터베이스입니다."
]
# 문서를 벡터화
vectors = [ model.encode(t).tolist() for t in texts]

# 5. Qdrant에 업로드
points = [ # 각 문서에 고유 ID, 벡터, 원문 텍스트(payload)를 함께 저장
    models.PointStruct(id=i, vector=vectors[i], payload={"text": texts[i]})
    for i in range(len(texts))
]
# upsert는 새 데이터를 삽입하거나, 기존 데이터를 업데이트 한다
client.upsert(collection_name=collection_name, points=points)

print("문서 벡터 저장 완료")

/var/folders/63/l37nw44j2rv3t67022wjm2b40000gn/T/ipykernel_25667/4150081488.py:15: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


문서 벡터 저장 완료


벡터 데이터 저장 위치
- Qdrant 컬렉션: 벡터는 documents라는 이름의 컬렉션 안에 저장
- 포인트(PointStruct): 각 문서는 id, vector, payload로 구성된 포인트로 저장
- id: 문서 고유 식별자 (예: 0, 1, 2)
- vector: 384차원 임베딩 벡터 (리스트 형태)
- payload: 원문 텍스트 같은 메타데이터
- 즉, Qdrant 내부적으로는 벡터 데이터베이스 엔진이 벡터를 인덱싱하고, 검색할 수 있도록 저장한다

In [11]:
# scroll 사용해서 저장된 벡터 확인: Qdrant 클라이언트에서 scroll 이나 search 실행
result = client.scroll(
    collection_name="documents", 
    limit=3,
    with_vectors=True # 벡터도 포함해서 가져오기
)
for r in result[0]:
    print(r.id, r.payload, len(r.vector))

0 {'text': '안녕하세요, 이것은 한국어 문서입니다.'} 384
1 {'text': 'Hello, this is an English document.'} 384
2 {'text': 'Qdrant는 벡터 데이터베이스입니다.'} 384


In [12]:
# search 사용해서 저장된 벡터 확인: Qdrant 클라이언트에서 scroll 이나 search 실행
query = "Qdrant는 무엇인가요?"
# 질문을 임베딩하여 벡터(384차원) 생성
query_vector = model.encode(query).tolist()

result = client.query_points(
    collection_name="documents", # 검색할 컬렉션 이름
    query = query_vector, # 질의 벡터를 직접 전달
    limit=3, # 가장 유사한 3개의 문서를 반환
    with_payload=True, # 검색 결과에 원문 텍스트(payload)를 포함
    with_vectors=False # 필요시 True 설정하여 벡터도 가져올 수 있음
)
# 결과 출력 r.id 문서의 고유 ID, r.payload 저장된 원문 텍스트, r.score 질의 벡터와 저장된 벡터 간의 유사도 점수
for r in result.points:
    print(r.id, r.payload, r.score)

2 {'text': 'Qdrant는 벡터 데이터베이스입니다.'} 0.697295
1 {'text': 'Hello, this is an English document.'} 0.17940575
0 {'text': '안녕하세요, 이것은 한국어 문서입니다.'} 0.17834318
